# 01 · Train the five students

`clean`, `mix10`, `mix25`, `mix50`, `pureA` — all on Cloud et al.'s recipe
(r=8, **α=8**, 3 epochs, lr 2e-4, linear, max_seq 500), the only configuration
known to transmit on Qwen2.5-7B (I7).

`clean` trains from `mix10_clean.jsonl`; notebook 00 asserted the three clean
files are byte-identical, so this one student is the clean counterpart for every
fraction.

> **Run this from a terminal, not from the browser.** ~3 GPU-hours, and a
> dropped websocket kills a browser-attached kernel:
> ```
> cd /workspace/subliminal-attrib
> nohup jupyter nbconvert --to notebook --execute --inplace \
>     --ExecutePreprocessor.timeout=-1 notebooks/pivot/01_train.ipynb \
>     > /workspace/train.log 2>&1 &
> tail -f /workspace/train.log
> ```

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib, subprocess
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

# WHICH code is running? Nothing else in this notebook would notice a `main`
# checkout until a missing file several cells in, and pivot and main answer
# different questions -- their results have to stay independently attributable.
# sys.path puts ROOT/src first so the working tree beats any installed copy;
# assert that rather than assume it.
BRANCH = subprocess.run(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
assert BRANCH == "pivot", f"expected the 'pivot' branch at {ROOT}, found {BRANCH!r}"
assert Path(config.__file__).resolve().is_relative_to(ROOT / "src"), (
    f"subattr is imported from {config.__file__}, not {ROOT / 'src'}"
)
assert config.REPO_ROOT == ROOT, f"REPO_ROOT is {config.REPO_ROOT}, not {ROOT}"

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"branch    {BRANCH}   {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
from subattr import train as tr
from subattr.cache import free_gpu, gpu_memory

RECIPE = "cloud"
STUDENTS = [
    ("clean", MIX / "mix10_clean.jsonl"),   # asserted identical to mix25/mix50_clean in 00
    ("mix10", MIX / "mix10_mixed.jsonl"),
    ("mix25", MIX / "mix25_mixed.jsonl"),
    ("mix50", MIX / "mix50_mixed.jsonl"),
    ("pureA", MIX / "pure_A.jsonl"),        # the ceiling arm
]
for name, path in STUDENTS:
    assert path.exists(), f"{path} is missing -- run 00_setup first"
print(f"{len(STUDENTS)} students, recipe={RECIPE}")

In [ ]:
trained = {}
for name, path in STUDENTS:
    print(f"\n{'=' * 70}\n{name}  <-  {path.name}\n{'=' * 70}", flush=True)
    trained[name] = tr.train_student(cfg, path, name=name, recipe=RECIPE)
    free_gpu()
    print(f"[gpu] {gpu_memory()}", flush=True)

## 1.1 · Verify what was actually trained

The recipe is applied inside `resolve_config` and so is **not** part of the run
hash — the marker is the only record that these are α=8 / 3-epoch students and
not the `spec` recipe. Assert it rather than trust the directory name.

In [ ]:
for name, student in trained.items():
    marker = json.loads((Path(student.adapter_dir) / "subattr_complete.json").read_text())
    assert marker["recipe"] == "cloud", f"{name} was trained with {marker['recipe']!r}"
    assert marker["num_train_epochs"] == 3
    assert marker["lora_alpha"] == 8
    assert marker["n_examples"] == 10000, f"{name} saw {marker['n_examples']} examples"
    print(f"  {name:<8s} {marker['n_examples']} examples, {marker['num_train_epochs']} epochs, "
          f"alpha={marker['lora_alpha']}  ->  {tr.latest_adapter(student)}")

In [ ]:
for name, student in trained.items():
    states = sorted(Path(student.adapter_dir).glob("**/trainer_state.json"))
    if not states:
        print(f"  {name:<8s} no trainer_state.json")
        continue
    history = json.loads(states[-1].read_text()).get("log_history", [])
    losses = [h["loss"] for h in history if "loss" in h]
    if losses:
        print(f"  {name:<8s} train loss {losses[0]:.4f} -> {losses[-1]:.4f}  ({len(losses)} logs)")

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.